In [23]:
import torch
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Количество GPU: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Текущий GPU: {torch.cuda.get_device_name(0)}")

PyTorch версия: 2.7.1+cu118
CUDA доступна: True
Количество GPU: 1
Текущий GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Работает как надо

In [3]:
import torch
import subprocess
import os
import time
import glob
import shutil

# ================== НАСТРОЙКИ ==================
INPUT_FILE = "data/1.mp3"
OUT_DIR = "data/1"
os.makedirs(OUT_DIR, exist_ok=True)

def check_gpu():
    """Проверка доступности GPU"""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # GB
        print(f"✅ GPU доступна: {gpu_name} ({gpu_memory:.1f} GB)")
        return True
    else:
        print("⚠️  GPU не доступна. Будет использоваться CPU (медленно)")
        return False

def separate_fast():
    """
    Быстрая версия с оптимизациями для RTX 3050
    """
    print("\n" + "="*60)
    print("⚡ ЗАПУСК УСКОРЕННОГО РАЗДЕЛЕНИЯ")
    print("="*60)
    
    has_gpu = torch.cuda.is_available()
    
    # Выбираем оптимальные параметры
    model = "htdemucs"  # самый быстрый
    
    cmd = [
        "demucs",
        "--two-stems", "vocals",      # отделяем вокал
        "-n", model,
        "--shifts", "1",              # 1 сдвиг для баланса скорости/качества
        "--clip-mode", "rescale",     # быстрее чем clamp
        "-o", OUT_DIR,
        INPUT_FILE
    ]
    
    # Добавляем GPU параметры если доступно
    if has_gpu:
        cmd.extend(["-d", "cuda"])
        # Для GPU добавляем дополнительные оптимизации
        cmd.extend(["--jobs", "2"])
    else:
        cmd.extend(["--jobs", str(max(1, os.cpu_count() // 2))])
    
    print(f"\n🔧 Параметры:")
    print(f"   Модель: {model}")
    print(f"   Устройство: {'GPU (CUDA)' if has_gpu else 'CPU'}")
    print(f"   Сдвиги: 1")
    print(f"   Файл: {INPUT_FILE}")
    print(f"   Выходная папка: {OUT_DIR}")
    
    print(f"\n▶️  Команда: {' '.join(cmd)}")
    print("\n⏳ Обработка началась...")
    
    start_time = time.time()
    
    try:
        # Запускаем с перехватом вывода
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        # Выводим прогресс в реальном времени
        print("\n" + "-"*40)
        for line in process.stdout:
            line = line.strip()
            # Показываем только важные сообщения
            if any(keyword in line.lower() for keyword in 
                   ["separating", "loading", "model", "time:", "done"]):
                print(f"  {line}")
            elif "%" in line or "it/s" in line:
                print(f"  📊 {line}")
        print("-"*40)
        
        process.wait()
        
        elapsed = time.time() - start_time
        
        if process.returncode == 0:
            print(f"\n✅ Завершено успешно!")
            print(f"⏱️  Время выполнения: {elapsed:.1f} секунд ({elapsed/60:.1f} минут)")
            
            # Копируем и показываем результаты
            copy_and_show_results(model)
        else:
            print(f"\n❌ Ошибка выполнения (код: {process.returncode})")
            
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Ошибка Demucs: {e}")
    except FileNotFoundError:
        print("\n❌ Demucs не найден. Установите:")
        print("   pip install demucs")
        if has_gpu:
            print("   pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118")
    except Exception as e:
        print(f"\n❌ Неожиданная ошибка: {e}")

def copy_and_show_results(model_name):
    """
    Копирует результаты в удобное место и показывает информацию
    """
    model_dir = os.path.join(OUT_DIR, model_name)
    
    if not os.path.exists(model_dir):
        print(f"⚠️  Папка модели не найдена: {model_dir}")
        return
    
    # Ищем последнюю обработанную папку
    folders = glob.glob(os.path.join(model_dir, "*"))
    if not folders:
        print("⚠️  Не найдены результаты обработки")
        return
    
    latest_folder = max(folders, key=os.path.getmtime)
    
    print(f"\n📁 Результаты сохранены в: {latest_folder}")
    
    # Копируем вокал и музыку
    stems = {
        "vocals": "vocal_separated.wav",
        "no_vocals": "music_separated.wav"
    }
    
    for stem, output_name in stems.items():
        src_file = os.path.join(latest_folder, f"{stem}.wav")
        dst_file = os.path.join(OUT_DIR, output_name)
        
        if os.path.exists(src_file):
            try:
                shutil.copy2(src_file, dst_file)
                size_mb = os.path.getsize(dst_file) / (1024 * 1024)
                duration = get_audio_duration(dst_file)
                
                if stem == "vocals":
                    print(f"\n🎤 ВОКАЛ сохранен:")
                else:
                    print(f"\n🎶 МУЗЫКА сохранена:")
                    
                print(f"   Файл: {output_name}")
                print(f"   Размер: {size_mb:.1f} MB")
                print(f"   Длительность: {duration}")
                
            except Exception as e:
                print(f"⚠️  Ошибка копирования {stem}: {e}")
        else:
            print(f"⚠️  Файл не найден: {src_file}")

def get_audio_duration(file_path):
    """
    Получает длительность аудио файла
    """
    try:
        import librosa
        duration = librosa.get_duration(filename=file_path)
        minutes = int(duration // 60)
        seconds = int(duration % 60)
        return f"{minutes}:{seconds:02d}"
    except:
        return "неизвестно"

def get_file_info():
    """
    Показывает информацию о входном файле
    """
    if os.path.exists(INPUT_FILE):
        size_mb = os.path.getsize(INPUT_FILE) / (1024 * 1024)
        print(f"📊 Входной файл: {os.path.basename(INPUT_FILE)}")
        print(f"   Размер: {size_mb:.1f} MB")
        
        try:
            import librosa
            duration = librosa.get_duration(filename=INPUT_FILE)
            minutes = int(duration // 60)
            seconds = int(duration % 60)
            print(f"   Длительность: {minutes}:{seconds:02d}")
        except:
            pass
    else:
        print(f"❌ Файл не найден: {INPUT_FILE}")
        return False
    return True

# Главная функция
if __name__ == "__main__":
    print("=" * 60)
    print("УСКОРЕНИЕ РАЗДЕЛЕНИЯ ВОКАЛА И МУЗЫКИ")
    print("=" * 60)
    
    # Проверяем файл
    if not get_file_info():
        exit(1)
    
    # Проверяем GPU
    check_gpu()
    
    print("\n" + "=" * 60)
    print("🚀 ЗАПУСК ОБРАБОТКИ")
    print("=" * 60)
    
    # Запускаем обработку
    separate_fast()
    
    print("\n" + "=" * 60)
    print("✨ ОБРАБОТКА ЗАВЕРШЕНА")
    print("=" * 60)
    
    # Показываем где искать результаты
    print(f"\n📂 Итоговые файлы находятся в папке:")
    print(f"   {OUT_DIR}")
    print("\n🎵 Файлы:")
    print(f"   • vocal_separated.wav - вокал")
    print(f"   • music_separated.wav - инструментал")
    
    if torch.cuda.is_available():
        print(f"\n💡 Для еще большей скорости используйте:")
        print(f"   --shifts 0 --float16")

УСКОРЕНИЕ РАЗДЕЛЕНИЯ ВОКАЛА И МУЗЫКИ
📊 Входной файл: 1.mp3
   Размер: 7.2 MB
   Длительность: 3:03
✅ GPU доступна: NVIDIA GeForce RTX 3050 Laptop GPU (4.0 GB)

🚀 ЗАПУСК ОБРАБОТКИ

⚡ ЗАПУСК УСКОРЕННОГО РАЗДЕЛЕНИЯ

🔧 Параметры:
   Модель: htdemucs
   Устройство: GPU (CUDA)
   Сдвиги: 1
   Файл: data/1.mp3
   Выходная папка: data/1

▶️  Команда: demucs --two-stems vocals -n htdemucs --shifts 1 --clip-mode rescale -o data/1 data/1.mp3 -d cuda --jobs 2

⏳ Обработка началась...

----------------------------------------


C:\Temp\ipykernel_12044\2042672690.py:186: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename=INPUT_FILE)


  Selected model is a bag of 1 models. You will see that many progress bars per track.
  Separating track data\1.mp3
  📊 0%|                                                                                  | 0.0/187.2 [00:00<?, ?seconds/s]
  📊 3%|██▎                                                                      | 5.85/187.2 [00:03<01:49,  1.66seconds/s]
  📊 6%|████▌                                                                    | 11.7/187.2 [00:03<00:48,  3.64seconds/s]
  📊 9%|█████▌                                                     | 17.549999999999997/187.2 [00:04<00:28,  5.89seconds/s]
  📊 12%|█████████▏                                                               | 23.4/187.2 [00:04<00:19,  8.31seconds/s]
  📊 16%|███████████▎                                                            | 29.25/187.2 [00:04<00:14, 10.73seconds/s]
  📊 19%|███████████                                                | 35.099999999999994/187.2 [00:04<00:11, 13.01seconds/s]
  📊 22%|███████████

C:\Temp\ipykernel_12044\2042672690.py:168: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename=file_path)
